# Uber Ride Price Prediction
**Tasks:**
1. Pre-process the dataset.
2. Identify outliers.
3. Check the correlation.
4. Implement linear regression and random forest regression models.
5. Evaluate the models and compare their respective scores (R2, RMSE, etc.).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Load the dataset
df = pd.read_csv('uber.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Data_set/uber.csv'

## 1. Pre-process the dataset

In [ ]:
# Check info and null values
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

# Drop rows with missing values
df = df.dropna()

# Convert datetime column to actual datetime objects
if 'pickup_datetime' in df.columns:
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
    # Extract useful features from datetime
    df['hour'] = df['pickup_datetime'].dt.hour
    df['day'] = df['pickup_datetime'].dt.day
    df['month'] = df['pickup_datetime'].dt.month
    df['year'] = df['pickup_datetime'].dt.year
    df['dayofweek'] = df['pickup_datetime'].dt.dayofweek
    
    # Drop the original datetime column and any unneeded ID columns
    cols_to_drop = ['pickup_datetime', 'key', 'Unnamed: 0']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df.head()

## 2. Identify outliers

In [ ]:
# Visualizing outliers for fare_amount
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['fare_amount'])
plt.title('Boxplot of Fare Amount')
plt.show()

# Removing obvious outliers
print("Shape before outlier removal:", df.shape)

# Filter fare amount (must be positive, removing crazy high outliers)
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 200)]

if 'passenger_count' in df.columns:
    df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]
    
# Remove coordinate outliers (valid lat/long ranges)
if 'pickup_longitude' in df.columns:
    df = df[(df['pickup_longitude'] > -180) & (df['pickup_longitude'] < 180)]
    df = df[(df['dropoff_longitude'] > -180) & (df['dropoff_longitude'] < 180)]
    df = df[(df['pickup_latitude'] > -90) & (df['pickup_latitude'] < 90)]
    df = df[(df['dropoff_latitude'] > -90) & (df['dropoff_latitude'] < 90)]

print("Shape after outlier removal:", df.shape)

## 3. Check the correlation

In [ ]:
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Display correlations with fare_amount specifically
print(correlation_matrix['fare_amount'].sort_values(ascending=False))

## 4. Implement linear regression and random forest regression models

In [ ]:
# Split the data
X = df.drop('fare_amount', axis=1)
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# Random Forest Regression
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

## 5. Evaluate the models and compare their respective scores

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}\n")

evaluate_model(y_test, lr_preds, "Linear Regression")
evaluate_model(y_test, rf_preds, "Random Forest Regression")

# Simple comparison chart
scores = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R2 Score': [r2_score(y_test, lr_preds), r2_score(y_test, rf_preds)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, lr_preds)), np.sqrt(mean_squared_error(y_test, rf_preds))]
})

print(scores)